# KoELECTRA ONNX → INT8 TFLite 변환 (검증된 최종 버전)

## 사전 준비
Google Drive에 아래 두 파일 업로드:
```
내 드라이브/vp_model/model.onnx        (약 1.7 MB)
내 드라이브/vp_model/model.onnx.data   (약 431 MB)
```
로컬 경로: `NLU/models/onnx/`

## 결과
- `model_dynamic_range_quant.tflite` — **108 MB** (가중치 INT8, 연산 float32)
- FlexErf 포함 → Android 배포 시 `tensorflow-lite-select-tf-ops` 의존성 필요

## 주요 트러블슈팅 메모
| 시도 | 결과 |
|------|------|
| onnxsim + overwrite_input_shapes | 상수 폴딩으로 2GB+ 팽창 → 사용 불가 |
| disable_group_convolution | SavedModel 생성 불가 (onnx2tf 버그) |
| TFLite 추론 in Colab | FlexErf로 실행 불가 → ONNX Runtime으로 대체 검증 |
| custom_input_op_name_np_data_path | `[name, path, mean, std]` 4개 필드 필수 |

In [ ]:
# ── Step 1: 패키지 설치 ──────────────────────────────────────────────────
# TensorFlow / protobuf 는 Colab 기본 버전 유지 (버전 핀 금지)
!pip install -q onnx2tf sng4onnx onnxsim onnx onnxruntime transformers
!pip install -q onnx-graphsurgeon          # onnx2tf 내부 의존성
!pip install -q --upgrade protobuf         # TF 2.18+ 호환 복구

print('설치 완료')
print('⚠️  반드시 런타임 재시작 후 Step 2 부터 실행하세요.')
print('   런타임 메뉴 → 세션 다시 시작 (이 셀은 재실행 불필요)')

In [ ]:
# ── Step 2: Google Drive 마운트 ──────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/vp_model'
ONNX_SRC  = f'{DRIVE_DIR}/model.onnx'
ONNX_DATA = f'{DRIVE_DIR}/model.onnx.data'

assert os.path.exists(ONNX_SRC),  f'파일 없음: {ONNX_SRC}'
assert os.path.exists(ONNX_DATA), f'파일 없음: {ONNX_DATA}'
print(f'model.onnx      : {os.path.getsize(ONNX_SRC)  / 1024**2:.1f} MB')
print(f'model.onnx.data : {os.path.getsize(ONNX_DATA) / 1024**2:.1f} MB')

In [ ]:
# ── Step 3: ONNX → INT8 TFLite 직변환 ───────────────────────────────────
# SavedModel 경로 불가 (onnx2tf GroupConv 버그)
# → output_integer_quantized_tflite=True 로 TFLite 직접 출력
# → 결과: model_dynamic_range_quant.tflite (~108MB)
import shutil, glob, os, io, contextlib
import numpy as np
from transformers import AutoTokenizer
import onnx2tf

TMP_DIR = '/tmp/onnx_src'
OUT_DIR = '/tmp/tflite_out'
os.makedirs(TMP_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)
shutil.copy(ONNX_SRC,  f'{TMP_DIR}/model.onnx')
shutil.copy(ONNX_DATA, f'{TMP_DIR}/model.onnx.data')
print('파일 복사 완료')

# 캘리브레이션 데이터 (mean=0, std=1 필수 — 정수 토큰 ID 는 정규화 없음)
tokenizer = AutoTokenizer.from_pretrained('monologg/koelectra-base-v3-discriminator')
CALIB_TEXTS = [
    '저는 서울중앙지검 수사관입니다. 안전계좌로 즉시 송금해 주세요.',
    '당신의 통장이 범죄에 연루되었습니다. 계좌를 동결해야 합니다.',
    '주민등록번호와 카드번호를 알려주시면 처리해 드리겠습니다.',
    '공탁금 500만 원을 입금하지 않으면 구속영장이 발부됩니다.',
    'OTP 번호를 불러주시면 바로 처리해 드리겠습니다.',
    '금융감독원에서 연락드립니다. 계좌 보호 조치가 필요합니다.',
    '검찰청 수사관 김철수입니다. 사건 조회가 필요합니다.',
    '범죄에 연루된 계좌가 있어 즉시 이체를 요청드립니다.',
    '오늘 날씨가 정말 좋네요.',
    '내일 점심 뭐 먹을까요?',
    '안녕하세요, 배달 왔는데 문 앞에 놓고 갈까요?',
    '이번 주 회의는 수요일 오후 3시입니다.',
]
encs = [tokenizer(t, return_tensors='np', truncation=True, max_length=128, padding='max_length')
        for t in CALIB_TEXTS]
np.save('/tmp/calib_input_ids.npy',      np.concatenate([e['input_ids']      for e in encs]).astype(np.int64))
np.save('/tmp/calib_attention_mask.npy', np.concatenate([e['attention_mask']  for e in encs]).astype(np.int64))
np.save('/tmp/calib_token_type_ids.npy', np.concatenate([e['token_type_ids']  for e in encs]).astype(np.int64))
print(f'캘리브레이션 샘플: {len(CALIB_TEXTS)}개')

print('onnx2tf INT8 변환 중 (수 분 소요)...')
log_buf = io.StringIO()
try:
    with contextlib.redirect_stderr(log_buf):
        onnx2tf.convert(
            input_onnx_file_path=f'{TMP_DIR}/model.onnx',
            output_folder_path=OUT_DIR,
            overwrite_input_shape=[
                'input_ids:1,128',
                'attention_mask:1,128',
                'token_type_ids:1,128',
            ],
            output_integer_quantized_tflite=True,
            quant_type='per_channel',
            # 형식: [op_name, npy_path, mean, std]  ← 4개 필드 필수
            custom_input_op_name_np_data_path=[
                ['input_ids',      '/tmp/calib_input_ids.npy',      0, 1],
                ['attention_mask', '/tmp/calib_attention_mask.npy', 0, 1],
                ['token_type_ids', '/tmp/calib_token_type_ids.npy', 0, 1],
            ],
            non_verbose=True,
        )
except SystemExit as e:
    print(f'SystemExit({e.code})')
    print(log_buf.getvalue()[-4000:])
except Exception as e:
    print(f'예외: {e}')
    print(log_buf.getvalue()[-4000:])

print('\n── 생성된 파일 ──')
for f in sorted(glob.glob(f'{OUT_DIR}/*.tflite')):
    print(f'  {os.path.basename(f):45s} {os.path.getsize(f)/1024**2:7.1f} MB')

In [ ]:
# ── Step 4: 추론 검증 (ONNX Runtime) ─────────────────────────────────────
# TFLite 모델에 FlexErf 포함 → Colab 에서 직접 실행 불가
# ONNX Runtime 으로 원본 모델 정확도 기준값 확인
import numpy as np, onnxruntime as ort

TEST_SENTENCE = '저는 서울중앙지검 수사관입니다. 안전계좌로 즉시 송금해 주세요.'
LABEL_COLS    = ['기관사칭', '금전요구', '개인정보']
THRESHOLD     = 0.5

enc = tokenizer(
    TEST_SENTENCE, return_tensors='np',
    truncation=True, max_length=128, padding='max_length'
)

session = ort.InferenceSession(
    f'{TMP_DIR}/model.onnx',
    providers=['CPUExecutionProvider']
)
input_names = {i.name for i in session.get_inputs()}
feed   = {k: v.astype(np.int64) for k, v in enc.items() if k in input_names}
logits = session.run(['logits'], feed)[0][0]
probs  = 1 / (1 + np.exp(-logits))

print(f'입력: "{TEST_SENTENCE}"')
for col, p in zip(LABEL_COLS, probs):
    print(f'  {col}: {p:.3f}  {"█" * int(p * 20)}')
detected = [c for c, p in zip(LABEL_COLS, probs) if p >= THRESHOLD]
print(f'판정: {detected or ["정상"]}')

print('\n⚠️  FlexErf 포함 모델 → Android build.gradle.kts 에 추가 필요:')
print('   implementation("org.tensorflow:tensorflow-lite-select-tf-ops:2.14.0")')

In [ ]:
# ── Step 5: Google Drive 저장 ─────────────────────────────────────────────
import shutil, glob, os

OUT_DIR   = '/tmp/tflite_out'
DRIVE_DIR = '/content/drive/MyDrive/vp_model'
os.makedirs(DRIVE_DIR, exist_ok=True)

saved = []
for src in sorted(glob.glob(f'{OUT_DIR}/*.tflite')):
    dst = f'{DRIVE_DIR}/{os.path.basename(src)}'
    shutil.copy(src, dst)
    saved.append(dst)
    print(f'저장: {dst}  ({os.path.getsize(dst)/1024**2:.1f} MB)')

print('\n다운로드 후 배치 위치:')
print('  NLU/models/tflite/model_dynamic_range_quant.tflite')
print('  client/assets/model_int8.tflite  (이름 변경 후 복사)')